<a href="https://colab.research.google.com/github/Yshnavii/yshnavii.github.io/blob/main/firstrag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain_community langchainhub chromadb

In [ ]:
!pip install langchain langchain-openai

In [ ]:
from google.colab import userdata
import os
os.environ['OPENAI_API_KEY'] = userdata.get('Test_key')


In [ ]:
!pip install -q langchain-groq


In [ ]:
os.environ["GROQ_API_KEY"] = userdata.get("gro")

In [ ]:
print("OpenAI:", os.environ.get("OPENAI_API_KEY", "Not found")[:10])
print("Groq:", os.environ.get("GROQ_API_KEY", "Not found")[:10])

In [ ]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("/content/drive/MyDrive/ML/CUSAT_info.txt")
documents = loader.load()

print(documents[0].page_content[:500])


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
splits = text_splitter.split_documents(documents)
print(splits[1])

In [ ]:
print(len(splits))

In [ ]:

!pip install -q langchain-huggingface
!pip install -q sentence-transformers


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings
)

In [ ]:
results = vectorstore.similarity_search(
    "What hostel facilities are available?",
    k=3
)

for doc in results:
    print(doc.page_content)
    print("-" * 50)

In [ ]:
print(vectorstore._collection.count())

In [ ]:
print(vectorstore._collection.get())

In [ ]:
print("\nCollection 1: ",vectorstore._collection.get(ids=['c7408fea-d5bb-46f3-a1b0-0f4750749495']))

In [ ]:
retriever = vectorstore.as_retriever()

In [ ]:
prompt_template = """
Answer the question using only the context below.

Context:
{context}

Question:
{question}

Answer:
"""

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile"
)

In [ ]:
question = "What hostel facilities are available?"

docs = vectorstore.similarity_search(question, k=3)

context = "\n\n".join([doc.page_content for doc in docs])

prompt = prompt_template.format(
    context=context,
    question=question
)

response = llm.invoke(prompt)

print(response.content)

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

In [ ]:
docs = retriever.invoke("What facilities are available in the hostels?")
print(docs[0].page_content)

In [ ]:
!pip list | grep langchain

In [ ]:
import langchain
print(langchain.__version__)

In [ ]:
print(type(vectorstore))
response = llm.invoke("What is CUSAT?")
print(response.content)

In [ ]:
def ask_rag(question):
    docs = vectorstore.similarity_search(question, k=5)

    context = "\n\n".join(doc.page_content for doc in docs)

    prompt = f"""
    You are a helpful assistant.

    Use only the provided context to answer the question.
    If the answer is not present in the context, say:
    "I could not find that information in the provided documents."
    Answer the question using ONLY the context below.

    Context:
    {context}

    Question:
    {question}
    """

    response = llm.invoke(prompt)
    return response.content


In [ ]:
print(ask_rag("How many hostels does CUSAT have?"))
print(ask_rag("What facilities are available in hostels?"))
print(ask_rag("What departments are available?"))

In [ ]:
print(ask_rag("Can you provide the contact details of Cusat?"))

In [ ]:
print(ask_rag("Is CUSAT a good university?"))

In [ ]:
print(ask_rag("Does it have good placement?"))

In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr

def chat(message, history):
    answer = ask_rag(message)
    return answer

demo = gr.ChatInterface(
    fn=chat,
    title="CUSAT RAG Assistant"
)

demo.launch(share=True)